# FloraBERT benchmark-integrity audit (Kaggle)

Run this notebook directly in a Kaggle notebook with Internet enabled. It is the Kaggle counterpart of `benchmark_integrity_audit_colab.ipynb`.

This notebook downloads the public Hugging Face maize NAM expression dataset and maize promoter files, then checks:

- exact sequence overlap;
- reverse-complement overlap;
- 0.8-identity cross-dataset clusters with MMseqs2;
- expression split leakage; and
- whether a strict inductive benchmark is established.

No Torch, Transformers, Accelerate, model weights, Kaggle datasets, or credentials are required. The two Hugging Face datasets are public.


## Kaggle paths and resource policy

The active dataset cache, combined FASTA, MMseqs2 database, and temporary files are kept under `/kaggle/temp` so `/kaggle/working` remains reserved for downloadable reports.

Set `FLORABERT_AUDIT_ROOT` before this cell if a different scratch path is required. Set `FLORABERT_AUDIT_REPORT_ROOT` if the final report directory should have another name. A 30 GB or larger RAM runtime is recommended for the combined cluster audit.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


repo_url = os.environ.get(
    'FLORABERT_REPO_URL',
    'https://github.com/gurveervirk/florabert.git',
)
repo_ref = os.environ.get(
    'FLORABERT_REPO_REF',
    'audit/benchmark-integrity',
)
repo_dir = Path(
    os.environ.get('FLORABERT_REPO_DIR', '/kaggle/temp/florabert')
).expanduser()
expected_commit = os.environ.get('FLORABERT_REPO_COMMIT', '').strip()
script_relative = Path(
    'scripts/0-data-loading-processing/audit_benchmark_integrity.py'
)
script_path = repo_dir / script_relative

if not script_path.is_file():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f'{repo_dir} exists but does not contain the expected audit script'
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'Cloning {repo_url} branch {repo_ref} to {repo_dir}')
    subprocess.run(
        [
            'git',
            'clone',
            '--branch',
            repo_ref,
            '--single-branch',
            '--depth',
            '1',
            repo_url,
            str(repo_dir),
        ],
        check=True,
    )

if not script_path.is_file():
    raise FileNotFoundError(f'Audit script is missing: {script_path}')

actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=str(repo_dir),
    text=True,
).strip()
if expected_commit and actual_commit != expected_commit:
    raise RuntimeError(
        f'Unexpected FloraBERT commit {actual_commit}; '
        f'expected {expected_commit}'
    )

audit_root = Path(
    os.environ.get(
        'FLORABERT_AUDIT_ROOT',
        '/kaggle/temp/florabert_benchmark_audit_mmseqs',
    )
).expanduser()
report_root = Path(
    os.environ.get(
        'FLORABERT_AUDIT_REPORT_ROOT',
        '/kaggle/working/florabert_benchmark_audit_mmseqs',
    )
).expanduser()

audit_root.mkdir(parents=True, exist_ok=True)
report_root.mkdir(parents=True, exist_ok=True)

print('Repository:', repo_dir.resolve())
print('Repository ref:', repo_ref)
print('Repository commit:', actual_commit)
print('Audit scratch root:', audit_root.resolve())
print('Report output root:', report_root.resolve())
print('Expression dataset:', os.environ.get('FLORABERT_EXPRESSION_DATASET', 'Gurveer05/maize-nam-gene-expression-data'))
print('Maize MLM dataset:', os.environ.get('FLORABERT_MLM_DATASET', 'Gurveer05/maize-promoter-sequences'))

memory = shutil.disk_usage('/kaggle/temp')
print(f'/kaggle/temp disk free: {memory.free / 1024**3:.1f} GiB')
subprocess.run(['free', '-h'], check=False)


In [ ]:
# Install only the lightweight public-dataset readers.
# Do not install Torch or model-training packages for this audit.

import importlib.util

requirements = {
    'huggingface_hub': 'huggingface_hub>=0.23',
    'datasets': 'datasets>=2.18',
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print('Installing:', missing)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *missing],
        check=True,
    )
else:
    print('Lightweight audit dependencies are already available.')

print('HF token supplied by environment:', bool(
    os.environ.get('HF_TOKEN', '').strip()
    or os.environ.get('HUGGINGFACE_HUB_TOKEN', '').strip()
))
print('The configured Hugging Face datasets are public; a token is optional.')


In [ ]:
# Install or locate the CPU MMseqs2 binary.
# The clustering command below does not require a GPU.

import urllib.request

mmseqs_bin = (
    os.environ.get('FLORABERT_MMSEQS_BIN', '').strip()
    or shutil.which('mmseqs')
)

if not mmseqs_bin:
    mmseqs_root = Path('/kaggle/temp/mmseqs')
    mmseqs_archive = Path('/kaggle/temp/mmseqs-linux-avx2.tar.gz')
    mmseqs_executable = mmseqs_root / 'bin' / 'mmseqs'

    if not mmseqs_executable.is_file():
        if not mmseqs_archive.is_file():
            print('Downloading MMseqs2 AVX2 binary...')
            urllib.request.urlretrieve(
                'https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz',
                mmseqs_archive,
            )
        subprocess.run(
            ['tar', '-xzf', str(mmseqs_archive), '-C', '/kaggle/temp'],
            check=True,
        )

    mmseqs_bin = str(mmseqs_executable)

mmseqs_bin = str(Path(mmseqs_bin).expanduser())
probe = subprocess.run(
    [mmseqs_bin, 'version'],
    capture_output=True,
    text=True,
)
if probe.returncode != 0:
    raise RuntimeError(
        'MMseqs2 preflight failed.\n'
        + probe.stdout
        + probe.stderr
    )

# Defaults are intended for a roughly 30 GiB runtime. Override these
# environment variables before this cell if the runtime has less RAM.
mmseqs_workflow = os.environ.get(
    'FLORABERT_MMSEQS_WORKFLOW',
    'easy-cluster',
).strip()
if mmseqs_workflow not in {'easy-cluster', 'easy-linclust'}:
    raise ValueError(f'Unsupported MMseqs2 workflow: {mmseqs_workflow}')

mmseqs_split_memory_limit = os.environ.get(
    'FLORABERT_MMSEQS_SPLIT_MEMORY_LIMIT',
    '20G',
).strip()
mmseqs_threads = int(os.environ.get('FLORABERT_MMSEQS_THREADS', '2'))
if mmseqs_threads < 1:
    raise ValueError('FLORABERT_MMSEQS_THREADS must be at least 1.')

os.environ['FLORABERT_RUN_MMSEQS'] = '1'
os.environ['FLORABERT_MMSEQS_BIN'] = mmseqs_bin
os.environ['FLORABERT_MMSEQS_WORKFLOW'] = mmseqs_workflow
os.environ['FLORABERT_MMSEQS_SPLIT_MEMORY_LIMIT'] = mmseqs_split_memory_limit
os.environ['FLORABERT_MMSEQS_THREADS'] = str(mmseqs_threads)

print('MMseqs2 path:', mmseqs_bin)
print(probe.stdout.strip())
print('FLORABERT_RUN_MMSEQS:', os.environ['FLORABERT_RUN_MMSEQS'])
print('MMseqs2 workflow:', mmseqs_workflow)
print('MMseqs2 split memory limit:', mmseqs_split_memory_limit)
print('MMseqs2 threads:', mmseqs_threads)


In [ ]:
# Run the exact/reverse-complement audit and the combined MMseqs2 audit.
# Output is streamed so an active long-running process remains visible.

environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
environment['FLORABERT_AUDIT_ROOT'] = str(audit_root)
environment['FLORABERT_RUN_MMSEQS'] = '1'
environment['FLORABERT_MMSEQS_BIN'] = mmseqs_bin
environment['FLORABERT_MMSEQS_WORKFLOW'] = mmseqs_workflow
environment['FLORABERT_MMSEQS_SPLIT_MEMORY_LIMIT'] = mmseqs_split_memory_limit
environment['FLORABERT_MMSEQS_THREADS'] = str(mmseqs_threads)

command = [
    sys.executable,
    str(script_path),
    '--audit-root',
    str(audit_root),
    '--run-mmseqs',
    '--mmseqs-bin',
    mmseqs_bin,
    '--mmseqs-workflow',
    mmseqs_workflow,
    '--mmseqs-split-memory-limit',
    mmseqs_split_memory_limit,
    '--mmseqs-threads',
    str(mmseqs_threads),
]

print('Running:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    cwd=str(repo_dir),
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
finally:
    process.stdout.close()

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'Benchmark audit failed with exit code {return_code}. '
        'Inspect the MMseqs2 output above for memory or disk errors.'
    )


In [ ]:
# Inspect the machine-readable audit verdict and compact report tables.

import csv
import json

summary_path = audit_root / 'audit_summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(f'Audit summary was not created: {summary_path}')

audit_summary = json.loads(summary_path.read_text())
print('Strict inductive benchmark ready:', audit_summary['strict_inductive_benchmark_ready'])
print()
print('Expression counts:')
print(json.dumps(audit_summary['expression_counts'], indent=2))
print()
print('Exact overlap:')
print(json.dumps(audit_summary['exact_overlap'], indent=2))
print()
print('Cluster audit:')
print(json.dumps(audit_summary['cluster_audit'], indent=2))
print()
print('Interpretation:')
print(audit_summary['interpretation'])

for filename in [
    'exact_overlap_summary.csv',
    'exact_overlap_by_expression_split.csv',
    'cluster_overlap.csv',
    'expression_cluster_split_leakage.csv',
]:
    path = audit_root / filename
    if not path.is_file():
        print(f'{filename}: not created')
        continue
    print(f'\n{filename}:')
    with path.open(newline='', encoding='utf-8') as handle:
        for row_index, row in enumerate(csv.DictReader(handle)):
            print(row)
            if row_index >= 9:
                print('...')
                break

print('Scratch outputs:')
for path in sorted(audit_root.iterdir()):
    print(' ', path.name)


In [ ]:
# Copy reports to /kaggle/working for download.
# Large raw caches, combined FASTA, and MMseqs2 temporary databases stay in /kaggle/temp.

report_names = [
    'audit_summary.json',
    'cluster_audit_summary.json',
    'exact_overlap_summary.csv',
    'exact_overlap_by_expression_split.csv',
    'exact_overlap_examples.tsv',
    'cluster_overlap.csv',
    'expression_cluster_split_leakage.csv',
    'expression_sequence_manifest.tsv',
]

copied_reports = []
for filename in report_names:
    source = audit_root / filename
    if source.is_file():
        destination = report_root / filename
        shutil.copy2(source, destination)
        copied_reports.append(destination)

print('Reports copied to:', report_root.resolve())
for path in copied_reports:
    print(' ', path.name)
print('Combined FASTA and MMseqs2 temporary files remain under:', audit_root.resolve())


## Interpretation and retry notes

- `FLORABERT_RUN_MMSEQS=1` is explicitly passed to the script.
- The first retry uses `easy-cluster`, the original audit workflow, with a configurable `20G` split-memory limit and two threads.
- If the process is killed, lower `FLORABERT_MMSEQS_SPLIT_MEMORY_LIMIT` to `12G` or `16G`, or use a larger-RAM runtime.
- If the AVX2 binary reports an illegal-instruction error, use the official SSE4.1 binary and set `FLORABERT_MMSEQS_BIN` to its executable.
- A nonzero exact-overlap count already prevents a strict no-overlap inductive claim. MMseqs2 adds the near-duplicate and split-leakage analysis; it does not erase exact overlap.
- The reports under `/kaggle/working/florabert_benchmark_audit_mmseqs` are the files to download and archive with the dataset revisions and code commit.
